<a href="https://colab.research.google.com/github/vnsantos/projeto_mrp_costing/blob/main/MRP_Costing_Inflightor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import io
import re
import numpy as np
from google.colab import files

# --- PARTE 1: FUNÇÕES DE PROCESSAMENTO INICIAL ---

def processar_listagem():
    print("--- Selecione o arquivo de LISTAGEM ---")
    uploaded = files.upload()
    if not uploaded: return pd.DataFrame()
    colunas_modelo = ['Cia', 'Cardapio', 'Resumo_Cardapio', 'Serviço', 'Class', 'Código_Inflightor', r'Produto\Receita', 'Classe', 'Ciclo', '%', 'Quantidade', 'UN', 'Custo', 'Vigência Inicio', 'Vigência Termino', 'Base de Custo', 'Yield_Listagem']
    df_final_listagem = pd.DataFrame()

    for filename, content in uploaded.items():
        df_raw = pd.read_excel(io.BytesIO(content), header=None)
        rows_list = []
        current_cia, current_classe_meta, current_ciclo, current_vig_ini, current_vig_fim, current_base_custo, current_cardapio, current_resumo, current_servico = [""]*9

        for i in range(len(df_raw)):
            col_a = str(df_raw.iloc[i, 0]).strip().upper() if pd.notnull(df_raw.iloc[i, 0]) else ""

            # Identifica e atualiza variáveis de controle de metadados baseadas na linha atual
            for c in range(len(df_raw.columns)):
                cell_val = str(df_raw.iloc[i, c]).strip().upper() if pd.notnull(df_raw.iloc[i, c]) else ""
                if "COMPANHIA" in cell_val: current_cia = str(df_raw.iloc[i, c+2]).strip() if c+2 < len(df_raw.columns) else ""
                elif "CLASSE" in cell_val and "CLASSE:" not in cell_val and col_a not in ["P", "R", "IT:"]: current_classe_meta = str(df_raw.iloc[i, c+1]).strip() if c+1 < len(df_raw.columns) else ""
                elif "SERVIÇO" in cell_val or "SERVICO" in cell_val: current_servico = str(df_raw.iloc[i, 4]).strip() if pd.notnull(df_raw.iloc[i, 4]) else ""
                elif cell_val == "CUSTO" and col_a not in ["P", "R", "IT:"]: current_base_custo = str(df_raw.iloc[i, c+1]).strip() if c+1 < len(df_raw.columns) else ""

            # CORREÇÃO DEFINITIVA: O texto "CICLO" sempre estará na coluna G (índice 6) e o seu valor desejado sempre estará na coluna H (índice 7)
            if len(df_raw.columns) > 7:
                cell_g_val = str(df_raw.iloc[i, 6]).strip().upper() if pd.notnull(df_raw.iloc[i, 6]) else ""
                if "CICLO" in cell_g_val:
                    current_ciclo = str(df_raw.iloc[i, 7]).strip() if pd.notnull(df_raw.iloc[i, 7]) else ""

            if col_a == "IT:":
                full_name = str(df_raw.iloc[i, 2]).strip()
                current_cardapio = re.sub(r'\)(?!.*\)).*$', ')', full_name).strip()
                current_resumo = re.split(r'\s*\(', current_cardapio)[0].strip()
                yield_val = df_raw.iloc[i, 5] if pd.notnull(df_raw.iloc[i, 5]) else None
                for look_ahead in range(1, 3):
                    if i + look_ahead >= len(df_raw): break
                    col_c_val = str(df_raw.iloc[i + look_ahead, 2]).strip()
                    if "VIGÊNCIA:" in col_c_val.upper() or "VIGENCIA:" in col_c_val.upper():
                        dates = re.findall(r'\d{2}/\d{2}/\d{2,4}', col_c_val)
                        if len(dates) >= 2: current_vig_ini, current_vig_fim = dates[0], dates[1]
                rows_list.append({'Cia': current_cia, 'Cardapio': current_cardapio, 'Resumo_Cardapio': current_resumo, 'Serviço': current_servico, 'Class': '', r'Produto\Receita': full_name, 'Classe': current_classe_meta, 'Ciclo': current_ciclo, 'Vigência Inicio': current_vig_ini, 'Vigência Termino': current_vig_fim, 'Base de Custo': current_base_custo, 'Yield_Listagem': yield_val})
            elif col_a in ["P", "R"]:
                yield_val = df_raw.iloc[i, 5] if pd.notnull(df_raw.iloc[i, 5]) else None
                rows_list.append({'Cia': current_cia, 'Cardapio': current_cardapio, 'Resumo_Cardapio': current_resumo, 'Serviço': current_servico, 'Class': col_a, 'Classe': current_classe_meta, 'Ciclo': current_ciclo, 'Vigência Inicio': current_vig_ini, 'Vigência Termino': current_vig_fim, 'Base de Custo': current_base_custo, 'Código_Inflightor': df_raw.iloc[i, 1], r'Produto\Receita': df_raw.iloc[i, 2], 'Quantidade': df_raw.iloc[i, 7], 'UN': df_raw.iloc[i, 8], 'Custo': df_raw.iloc[i, 9], '%': df_raw.iloc[i, 11], 'Yield_Listagem': yield_val})

        df_temp = pd.DataFrame(rows_list)
        if r'Produto\Receita' in df_temp.columns: df_temp[r'Produto\Receita'] = df_temp[r'Produto\Receita'].astype(str).str.split('(').str[0].str.strip()
        for col in colunas_modelo:
            if col not in df_temp.columns: df_temp[col] = ""
        df_final_listagem = pd.concat([df_final_listagem, df_temp[colunas_modelo]])
    return df_final_listagem

def processar_receita():
    print("\n--- Selecione o arquivo de RECEITA ---")
    uploaded = files.upload()
    if not uploaded: return pd.DataFrame()
    file_name = list(uploaded.keys())[0]
    df = pd.read_excel(io.BytesIO(uploaded[file_name]), header=None, engine="openpyxl")
    df["ordem"] = df.index

    def buscar_valor(campo):
        for i in range(len(df)):
            for j in range(len(df.columns)):
                if str(df.iloc[i,j]).strip() == campo:
                    for col in range(j+1,len(df.columns)):
                        v = df.iloc[i,col]
                        if pd.notnull(v): return v
        return None

    unidade_base, menu_grid_cod, data_custo, companhia, menu_grid = buscar_valor("Unidade"), buscar_valor("Cod. Menu Grid"), buscar_valor("Data de Custo"), buscar_valor("Companhia"), buscar_valor("Menu Grid")

    is_new_recipe = (df.iloc[:, 0].isnull() | (df.iloc[:, 0].astype(str).str.strip() == "")) & (df.iloc[:, 1].notnull())
    df["Grupo_Receita"] = is_new_recipe.cumsum()
    df["Nome_Receita_Pai"] = df.iloc[:, 2].where(is_new_recipe)
    df["Agrupador_Receita"] = df.groupby("Grupo_Receita")["Nome_Receita_Pai"].transform("first")

    # Mapear e carregar os rendimentos de cada bloco para repassar a todos os filhos do grupo
    df["Rendimento_Liq"] = pd.NA
    df["Rendimento_KG"] = pd.NA
    df["Rendimento_Unidade"] = pd.NA
    df["Custo_Rendimento"] = pd.NA

    for g_id, bloco in df.groupby("Grupo_Receita"):
        if g_id == 0: continue
        last_liq, last_kg, last_unit, last_cost = [pd.NA]*4
        for idx in bloco.index:
            texto_a = str(bloco.loc[idx, 0]).upper()
            valor_e = pd.to_numeric(bloco.loc[idx, 4], errors="coerce")

            if "RENDIMENTO L" in texto_a and "RENDIMENTO LT" not in texto_a and pd.notnull(valor_e):
                last_liq = valor_e
            if any(term in texto_a for term in ["RENDIMENTO KG", "RENDIMENTO UN", "RENDIMENTO LT"]) and pd.notnull(valor_e):
                last_kg = valor_e
                last_unit = bloco.loc[idx, 5]
                last_cost = bloco.loc[idx, 8]

        # Propaga os rendimentos localizados para todas as linhas pertencentes a este mesmo grupo de receita
        df.loc[bloco.index, "Rendimento_Liq"] = last_liq
        df.loc[bloco.index, "Rendimento_KG"] = last_kg
        df.loc[bloco.index, "Rendimento_Unidade"] = last_unit
        df.loc[bloco.index, "Custo_Rendimento"] = last_cost

    df["Class"] = df.iloc[:,0].where(df.iloc[:,0].isin(["P","R","S"]))
    df_base = df[df.iloc[:,1].notnull()].copy()

    def capturar_dados(idx):
        res, línea = {"Quantidade": None, "UN": None, "Custo_Unit": None, "Yield": None, "Custo_c_Yield": None, "Custo_Total": None}, df.loc[idx]
        if pd.notnull(línea[8]) or pd.notnull(línea[5]): res.update(
            {"Quantidade": línea[8], "UN": línea[9], "Custo_Unit": línea[5], "Yield": línea[6], "Custo_c_Yield": línea[7], "Custo_Total": línea[10]}
        )
        return pd.Series(res)

    df_base = df_base.join(df.apply(lambda x: capturar_dados(x.name), axis=1, result_type="expand").loc[df_base.index])
    df_base["qtd_num"], df_base["rend_liq_num"], df_base["rend_kg_num"] = pd.to_numeric(df_base["Quantidade"], errors="coerce"), pd.to_numeric(df_base["Rendimento_Liq"], errors="coerce"), pd.to_numeric(df_base["Rendimento_KG"], errors="coerce")

    # Ajustado para considerar rend_kg_num (ou Rendimento LT) como divisor quando rend_liq_num não existir/for nulo ou zero
    df_base["Qtd_Unt_Calculada"] = np.where(
        df_base["Class"].notnull(),
        np.where(
            df_base["rend_liq_num"].notnull() & (df_base["rend_liq_num" ] != 0),
            (df_base["qtd_num"] / df_base["rend_liq_num"].replace(0, np.nan)) * df_base["rend_kg_num"],
            np.where(
                df_base["rend_kg_num"].notnull() & (df_base["rend_kg_num"] != 0),
                df_base["qtd_num"] / df_base["rend_kg_num"],
                df_base["qtd_num"]
            )
        ),
        np.nan
    )

    df_final_receita = pd.DataFrame({"Unidade": unidade_base, "Cod Menu Grid": menu_grid_cod, "Data Custo": data_custo, "Companhia": companhia, "Menu Grid": menu_grid, "Class": df_base["Class"], "Código": df_base.iloc[:,1], "Receita Principal": df_base["Agrupador_Receita"], "Produto/Receita": df_base.iloc[:,2], "Yield": df_base["Yield"], "Quantidade": np.where(df_base["Class"].isnull(), np.nan, df_base["qtd_num"]), "Qtd Unt": df_base["Qtd_Unt_Calculada"], "UN": df_base["UN"], "Rendimento_Liq": df_base["Rendimento_Liq"], "Rendimento_KG": df_base["Rendimento_KG"], "Unidade Rendimento": df_base["Rendimento_Unidade"], "Custo_Unit": df_base["Custo_Unit"], "Custo c Yield": df_base["Custo_c_Yield"], "Custo": df_base["Custo_Rendimento"], "ordem": df_base["ordem"]})
    df_final_receita = df_final_receita[~((df_final_receita["Produto/Receita"].astype(str).str.contains("Inflação|Total|Rendimento|Histórico")) & df_final_receita["Class"].notnull())]
    return df_final_receita.sort_values("ordem").drop(columns=["ordem"])

# --- EXECUÇÃO E UNIFICAÇÃO ---

df_listagem_proc = processar_listagem()
df_receita_proc = processar_receita()

if not df_listagem_proc.empty and not df_receita_proc.empty:
    df_listagem_proc["produto_norm"] = df_listagem_proc[r"Produto\Receita"].astype(str).str.strip().str.upper()
    df_receita_proc["principal_norm"] = df_receita_proc["Receita Principal"].astype(str).str.strip().str.upper()
    df_receita_proc["receita_norm"] = df_receita_proc["Produto/Receita"].astype(str).str.strip().str.upper()

    receita_dict = {k: v.to_dict('records') for k, v in df_receita_proc.groupby("principal_norm")}
    resultado_final = []

    def explodir_arvore(receita_norm, row_listagem, visitadas, active_multiplier=None):
        itens = receita_dict.get(receita_norm)
        if itens is None or receita_norm in visitadas: return
        visitadas.add(receita_norm)
        for item in itens:
            item_qtd = pd.to_numeric(item['Quantidade'], errors='coerce')
            rend_liq = pd.to_numeric(item['Rendimento_Liq'], errors='coerce')
            rend_kg = pd.to_numeric(item['Rendimento_KG'], errors='coerce')
            custo_unit = pd.to_numeric(item['Custo_Unit'], errors='coerce')

            if pd.notnull(rend_liq) and rend_liq != 0:
                base_qty = item_qtd / rend_liq
            elif pd.notnull(rend_kg) and rend_kg != 0:
                base_qty = item_qtd / rend_kg
            else:
                base_qty = item_qtd

            if item['Class'] == "R":
                qtd_calculada = base_qty * active_multiplier if active_multiplier is not None else base_qty
                new_multiplier = qtd_calculada
            elif item['Class'] == "S":
                qtd_calculada = base_qty * active_multiplier if active_multiplier is not None else base_qty
                new_multiplier = qtd_calculada
            else:
                if active_multiplier is not None:
                    qtd_calculada = base_qty * active_multiplier
                else:
                    if pd.notnull(rend_liq) and rend_liq != 0:
                        qtd_calculada = (item_qtd / rend_liq)
                    elif pd.notnull(rend_kg) and rend_kg != 0:
                        qtd_calculada = (item_qtd / rend_kg)
                    else:
                        qtd_calculada = item_qtd
                new_multiplier = active_multiplier

            if active_multiplier is not None:
                custo_calc_unid = (custo_unit * qtd_calculada) if pd.notnull(custo_unit) and pd.notnull(qtd_calculada) else 0
            else:
                custo_c_yield = pd.to_numeric(item['Custo c Yield'], errors='coerce')
                custo_calc_unid = (custo_c_yield * qtd_calculada) if pd.notnull(custo_c_yield) and pd.notnull(qtd_calculada) else 0

            resultado_final.append((
                item['Unidade'], item['Cod Menu Grid'], row_listagem['Base de Custo'], row_listagem['Cia'],
                row_listagem['Serviço'], row_listagem['Cardapio'], 'Rec', item['Class'], item['Código'],
                item['Produto/Receita'], row_listagem['Classe'], row_listagem['Ciclo'], row_listagem['Quantidade'],
                row_listagem['%'], row_listagem['UN'], row_listagem['Custo'], item['Yield'],
                qtd_calculada, item['Qtd Unt'], item['UN'], item['Rendimento_Liq'], item['Rendimento_KG'],
                item['Custo_Unit'], item['Custo c Yield'], item['Custo'], row_listagem['Yield_Listagem'], custo_calc_unid
            ))

            if item['Class'] in ["R", "S"]:
                explodir_arvore(item['receita_norm'], row_listagem, visitadas.copy(), active_multiplier=new_multiplier)

    for row in df_listagem_proc.to_dict('records'):
        if not row['Class'] or pd.isna(row['Class']) or str(row['Class']).strip() == "":
            resultado_final.append((None, None, row['Base de Custo'], row['Cia'], row['Serviço'], row['Cardapio'], 'List', row['Class'], row['Código_Inflightor'], row[r'Produto\Receita'], row['Classe'], row['Ciclo'], 1.0, row['%'], 'UN', 0.0, None, 1.0, None, 'UN', None, None, None, None, None, row['Yield_Listagem'], 0.0))
        else:
            resultado_final.append((None, None, row['Base de Custo'], row['Cia'], row['Serviço'], row['Cardapio'], 'List', row['Class'], row['Código_Inflightor'], row[r'Produto\Receita'], row['Classe'], row['Ciclo'], row['Quantidade'], row['%'], row['UN'], row['Custo'], None, row['Quantidade'], None, None, None, None, None, None, None, row['Yield_Listagem'], row['Custo']))

        if row['Class'] != "P":
            listagem_qty = pd.to_numeric(row['Quantidade'], errors='coerce')
            explodir_arvore(row['produto_norm'], row, set(), active_multiplier=listagem_qty)

    colunas = [
        "Base", "Cod Menu Grid", "Base de custo", "Cia", "Serviço", "Cardapio",
        "Relatorio", "Class", "Codigo inflightor", "Produto e receita", "Classe", "Ciclo",
        "Qtd", "%", "UN", "CustoL", "Yield", "Quantidade", "Qtd Unt", "Unidade",
        "Rendimento Liquido", "Rendimento KG", "Custo Unit", "Custo Total", "Custo", "Yield_Listagem", "Custo Unitario"
    ]

    df_final = pd.DataFrame(resultado_final, columns=colunas)

    # CORREÇÃO DA EXTRAÇÃO DE %: Usando findall para pegar sempre a última aparição de parênteses
    def extract_pct(val):
        matches = re.findall(r'\(([^)]+)\)', str(val))
        return matches[-1] if matches else np.nan

    df_final['pct_temp'] = df_final['Cardapio'].apply(extract_pct)
    df_final['pct_temp'] = df_final['pct_temp'].ffill()
    df_final['%'] = df_final['pct_temp'].fillna(df_final['%'])
    df_final['Yield'] = df_final['Yield'].fillna(df_final['Yield_Listagem'])
    df_final['Yield'] = df_final['Yield'].apply(lambda x: np.nan if isinstance(x, str) and 'RENDIMENTO' in x.upper() else x)
    df_final = df_final[~((df_final['Class'].isna() | (df_final['Class'].astype(str).str.strip() == "")) & (df_final['Codigo inflightor'].notna() & (df_final['Codigo inflightor'].astype(str).str.strip() != "")))]
    df_final['Unidade'] = df_final['Unidade'].fillna(df_final["UN"])
    df_final[['Cardapio','Base','Cod Menu Grid']] = df_final[['Cardapio','Base','Cod Menu Grid']].ffill()

    colunas_para_remover = [
        'Qtd', 'UN', 'CustoL', 'pct_temp', 'Yield_Listagem',
        'Rendimento Liquido', 'Rendimento KG', 'Custo Unit', 'Custo Total', 'Custo', 'Qtd Unt'
    ]
    df_final.drop(columns=colunas_para_remover, inplace=True, errors='ignore')

    df_final['Custo Unitario'] = pd.to_numeric(df_final['Custo Unitario'], errors='coerce').fillna(0)

    df_final['temp_servico'] = df_final['Serviço'].astype(str).str.strip().str.upper()
    df_final['temp_cardapio'] = df_final['Cardapio'].astype(str).str.strip().str.upper()
    df_final['temp_classe'] = df_final['Classe'].astype(str).str.strip().str.upper()
    df_final['temp_ciclo'] = df_final['Ciclo'].astype(str).str.strip().str.upper()
    df_final['temp_relatorio'] = df_final['Relatorio'].astype(str).str.strip().str.upper()

    criterios_somases = ['temp_servico', 'temp_cardapio', 'temp_classe', 'temp_ciclo', 'temp_relatorio']
    df_final['Custo_Somases_Calculado'] = df_final.groupby(criterios_somases)['Custo Unitario'].transform('sum')

    df_final['Custo Unitario'] = np.where(
        (df_final['Relatorio'] == 'List') & (df_final['Class'].isna() | (df_final['Class'].astype(str).str.strip() == '')),
        df_final['Custo_Somases_Calculado'],
        df_final['Custo Unitario']
    )

    df_final.drop(columns=criterios_somases + ['Custo_Somases_Calculado'], inplace=True, errors='ignore')

    # NOVA REGRA DE NOMENCLATURA SIMPLIFICADA:
    # Apenas o Cia (Coluna D, ex: 'Cia')
    val_cia = str(df_final['Cia'].iloc[0]).strip() if len(df_final) > 0 and pd.notnull(df_final['Cia'].iloc[0]) else ""

    # Substituir possíveis caracteres inválidos em nomes de arquivos do sistema operacional por sublinhados
    def limpar_nome_arquivo(txt):
        return re.sub(r'[\//*?:"<>|]', '_', txt)

    suffix_name = limpar_nome_arquivo(val_cia).strip()
    saida = f"MRP Costing {suffix_name}.xlsx" if suffix_name else "MRP Costing.xlsx"

    with pd.ExcelWriter(saida, engine='openpyxl') as writer:
        df_final.to_excel(writer, sheet_name="Dados", index=False)
        ws = writer.sheets["Dados"]
        for r in range(2, ws.max_row + 1):
            col_pct = df_final.columns.get_loc("%") + 1 if "%" in df_final.columns else 12
            col_yield = df_final.columns.get_loc("Yield") + 1 if "Yield" in df_final.columns else 13
            col_custo = df_final.columns.get_loc("Custo Unitario") + 1 if "Custo Unitario" in df_final.columns else 17

            ws.cell(row=r, column=col_pct).number_format = '0.00%'
            ws.cell(row=r, column=col_yield).number_format = '0.00%'
            ws.cell(row=r, column=col_custo).number_format = 'R$ #,##0.00'

    print(f'\nSucesso! Arquivo unificado gerado como: {saida}')
    files.download(saida)

--- Selecione o arquivo de LISTAGEM ---



--- Selecione o arquivo de RECEITA ---
